# Figure. Demographic Profile of Maricopa County versus U.S. National Average

## Purpose
Creates the demographic benchmark table used in the manuscript.

## Data
- **U.S. comparison:** IPUMS ACS person-level extract (`usa_00002.csv.gz`)
- **Maricopa County values:** the values already used in the working analysis from the Maricopa County economic profile
- To find the calues that were used for Maricopa, you can find them in the CFA Landscape Workfing Draft that Rachel and Mike found. 
- Look for: economic-profile-maricopacounty (1).pdf
## Measures
- Individuals with a disability
- Veterans
- Uninsured individuals
- Population below 100% of the poverty line
- Adults without a high school diploma

This figure is demographic rather than sector-specific, so the Healthcare / Manufacturing / Construction / Semiconductor ordering does not apply.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

COMPUTER = "W"

if str(COMPUTER).strip().upper() == "W":
    IPUMS_FILE = Path(
        r"C:\Users\301533\Documents\Walmart_Analysis_Output"
        r"\SOCS& NAICS\usa_00002.csv.gz"
    )
elif str(COMPUTER).strip().upper() == "P":
    IPUMS_FILE = Path(
        r"C:\Users\Arielle\Desktop\Desktop\New folder\Walmart"
        r"\usa_00002.csv.gz"
    )
else:
    raise ValueError('COMPUTER must be "W" or "P".')

OUTPUT_FOLDER = Path(
    r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)

if not IPUMS_FILE.exists():
    raise FileNotFoundError(
        f"IPUMS file not found:\n{IPUMS_FILE}\n"
        "Update IPUMS_FILE if the extract is stored elsewhere."
    )

print("IPUMS source:", IPUMS_FILE)


IPUMS source: C:\Users\301533\Documents\Walmart_Analysis_Output\SOCS& NAICS\usa_00002.csv.gz


In [2]:
# ============================================================
# CALCULATE U.S. BENCHMARKS FROM IPUMS
# ============================================================

tot_pop_wt = 0
adult_pop_wt = 0
disability_wt = 0
veteran_wt = 0
uninsured_wt = 0
poverty_wt = 0
no_hs_wt = 0

for chunk in pd.read_csv(
    IPUMS_FILE,
    compression="gzip",
    chunksize=200_000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.upper().str.strip()

    if "PERWT" not in chunk.columns:
        raise KeyError("IPUMS extract must contain PERWT.")

    chunk["PERWT"] = pd.to_numeric(
        chunk["PERWT"],
        errors="coerce"
    ).fillna(0)

    tot_pop_wt += chunk["PERWT"].sum()

    age = pd.to_numeric(
        chunk["AGE"],
        errors="coerce"
    )
    adult = age.ge(18)

    adult_pop_wt += chunk.loc[adult, "PERWT"].sum()

    diff_cols = [
        c for c in [
            "DIFFREM", "DIFFPHYS", "DIFFMOB", "DIFFCARE",
            "DIFFSENS", "DIFFEYE", "DIFFHEAR"
        ]
        if c in chunk.columns
    ]

    if diff_cols:
        disability = (
            chunk[diff_cols]
            .apply(pd.to_numeric, errors="coerce")
            .eq(2)
            .any(axis=1)
        )
        disability_wt += chunk.loc[disability, "PERWT"].sum()

    if "VETSTAT" in chunk.columns:
        vet = (
            pd.to_numeric(chunk["VETSTAT"], errors="coerce").eq(2)
            & adult
        )
        veteran_wt += chunk.loc[vet, "PERWT"].sum()

    if "HCOVANY" in chunk.columns:
        uninsured = pd.to_numeric(
            chunk["HCOVANY"],
            errors="coerce"
        ).eq(1)
        uninsured_wt += chunk.loc[uninsured, "PERWT"].sum()

    if "POVERTY" in chunk.columns:
        poverty = pd.to_numeric(
            chunk["POVERTY"],
            errors="coerce"
        )
        poor = poverty.gt(0) & poverty.lt(100)
        poverty_wt += chunk.loc[poor, "PERWT"].sum()

    if "EDUC" in chunk.columns:
        educ = pd.to_numeric(
            chunk["EDUC"],
            errors="coerce"
        )
        no_hs = educ.gt(0) & educ.lt(6) & adult
        no_hs_wt += chunk.loc[no_hs, "PERWT"].sum()

us = {
    "Individuals with a Disability": disability_wt / tot_pop_wt,
    "Veterans": veteran_wt / adult_pop_wt,
    "Uninsured Individuals": uninsured_wt / tot_pop_wt,
    "Low-Income Population (<100% Poverty)": poverty_wt / tot_pop_wt,
    "Adults Without High School Diploma": no_hs_wt / adult_pop_wt,
}

# Values already used in the manuscript analysis.
maricopa = {
    "Individuals with a Disability": 0.122,
    "Veterans": 0.066,
    "Uninsured Individuals": 0.106,
    "Low-Income Population (<100% Poverty)": 0.075,
    "Adults Without High School Diploma": 0.047,
}

result = pd.DataFrame({
    "Demographic Group": list(maricopa.keys()),
    "Maricopa County Share": list(maricopa.values()),
    "U.S. National Average": [us[k] for k in maricopa],
})

result["Difference (percentage points)"] = (
    100 * (
        result["Maricopa County Share"]
        - result["U.S. National Average"]
    )
)

display(result)


,Demographic Group,Maricopa County Share,U.S. National Average,Difference (percentage points)
0,Individuals with a Disability,0.122,0.137385,-1.538525
1,Veterans,0.066,0.061934,0.406576
2,Uninsured Individuals,0.106,0.086202,1.979816
3,Low-Income Population (<100% Poverty),0.075,0.106372,-3.137195
4,Adults Without High School Diploma,0.047,0.066971,-1.997146


In [3]:
# ============================================================
# EXPORT
# ============================================================

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_FOLDER
    / "Figure_Demographic_Profile_Maricopa_vs_US.xlsx"
)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Demographic Profile",
        index=False
    )

    ws = writer.book["Demographic Profile"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False

print("Saved:", OUTPUT_FILE)


Saved: G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis\SOC NAICS List\Excel Sheets\Arielle Tables\Figure_Demographic_Profile_Maricopa_vs_US.xlsx
